# Modelo de Estoque Inteligente do Canil

Este notebook explica o modelo usado para prever consumo de **racao**, **vacinas** e **agua sanitaria** no canil.

A entrega principal do modelo nao e apenas prever consumo, mas transformar essa previsao em uma decisao operacional:

- quantos dias de estoque restam;
- qual item esta critico;
- qual item deve ser comprado primeiro;
- quanto comprar como reposicao sugerida.

O modelo atual e um baseline supervisionado de regressao chamado `InventoryConsumptionModel`, implementado em `analytics/models.py`.

## 1. Tipo de modelo usado

O modelo usa **Ridge Regression**, uma regressao linear com regularizacao.

Em termos simples, ele aprende uma relacao entre variaveis do canil e o consumo diario de cada item:

```text
consumo_diario = intercepto + peso_1 * variavel_1 + peso_2 * variavel_2 + ...
```

A regularizacao Ridge reduz pesos exagerados e ajuda o modelo a ficar mais estavel, principalmente quando ainda temos poucos dados reais.

O pipeline treina um modelo separado para cada alvo:

- `feed_kg_consumed`: racao consumida em kg por dia;
- `vaccine_doses_used`: doses de vacina usadas por dia;
- `bleach_liters_used`: agua sanitaria usada em litros por dia.

## 2. Variaveis de entrada

| Variavel | Significado | Impacto esperado |
|---|---|---|
| `dogs_small` | quantidade de caes pequenos no canil | aumenta consumo de racao e limpeza |
| `dogs_medium` | quantidade de caes medios no canil | aumenta consumo de racao e limpeza |
| `dogs_large` | quantidade de caes grandes no canil | aumenta bastante consumo de racao |
| `cats` | quantidade de gatos no canil | aumenta racao de gatos, vacinas felinas e limpeza |
| `equine_vaccine_visits` | equinos trazidos somente para vacinacao externa | aumenta apenas consumo de vacinas |
| `puppies` | filhotes recebidos ou presentes no fluxo | aumenta demanda de vacinas, pois filhotes tomam 3 doses |
| `adult_new_animals` | animais adultos novos | aumenta vacinas iniciais, pois adultos novos tomam 2 doses |
| `resident_adults` | adultos residentes no canil | aumenta reforcos anuais |
| `quarantine_animals` | animais em quarentena | aumenta limpeza e pode aumentar atencao sanitaria |
| `cleaning_runs` | quantidade de ciclos de limpeza no dia | aumenta consumo de agua sanitaria |
| `outbreak_alert` | alerta de surto/doenca | aumenta vacinas e limpeza |
| `weekday_sin`, `weekday_cos` | sazonalidade semanal | captura diferenca entre dias uteis e fim de semana |
| `month_sin`, `month_cos` | sazonalidade anual | captura meses com maior entrada/campanha |

Observacao importante: **equinos nao ficam no canil**. Eles aparecem no modelo apenas como `equine_vaccine_visits`, ou seja, atendimentos externos de vacinacao.

## 3. Regras de vacina usadas no gerador artificial

As regras abaixo foram convertidas em logica no gerador de dados artificiais:

- caes filhotes: 3 doses;
- caes adultos novos: 2 doses;
- caes adultos residentes: 1 dose anual;
- felinos filhotes: 3 doses;
- felinos adultos novos: 2 doses;
- felinos adultos residentes: 1 dose anual;
- equinos: atendimento externo, 1 dose anual.

Vacinas consideradas:

- caninas: cinomose, hepatite canina, parvovirose canina, coronavirus canino, infeccoes respiratorias por adenovirus tipo 2;
- felinas: V5;
- equinas: influenza equina, rinopneumonite equina, encefalomielite equina e tetano.

In [3]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analytics").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analytics.models import InventoryConsumptionModel
from analytics.synthetic_data import generate_default_datasets
from analytics.data_pipeline import build_forecast_payload

pd.set_option("display.max_columns", 40)

## 4. Gerando uma base artificial de demonstracao

Enquanto o canil ainda nao possui historico real organizado, usamos uma base artificial para demonstrar o funcionamento.

Quando existirem dados reais, a ideia e substituir esta geracao por uma extracao do banco de dados do sistema.

In [4]:
history_df, future_df = generate_default_datasets(history_days=420, forecast_days=30, seed=42)

print(f"Historico: {history_df.shape[0]} linhas")
print(f"Futuro para previsao: {future_df.shape[0]} linhas")

history_df.head(10)

Historico: 420 linhas
Futuro para previsao: 30 linhas


,date,dogs_small,dogs_medium,dogs_large,cats,equine_vaccine_visits,puppies,adult_new_animals,resident_adults,quarantine_animals,cleaning_runs,outbreak_alert,feed_kg_consumed,vaccine_doses_used,bleach_liters_used
0,2025-03-28,24,54,28,19,1,2,4,123,12,7,0,32.89,1,14.32
1,2025-03-29,24,54,28,19,0,0,3,125,12,7,0,33.00,0,12.68
2,2025-03-30,26,57,29,19,0,0,6,131,13,8,0,35.78,0,14.90
3,2025-03-31,26,57,29,21,0,1,3,132,12,7,0,34.63,0,12.43
4,2025-04-01,27,58,30,22,0,2,5,135,14,9,0,35.74,1,16.92
5,2025-04-02,26,57,30,23,1,0,4,136,11,8,0,34.58,0,14.98
6,2025-04-03,27,58,31,23,1,2,3,137,11,8,0,34.17,3,14.74
7,2025-04-04,27,58,31,24,0,1,5,139,9,8,0,35.70,1,16.09
8,2025-04-05,27,58,31,23,0,0,2,139,9,8,0,35.36,1,14.32
9,2025-04-06,29,61,33,24,0,1,8,146,18,9,0,38.27,1,18.63


## 5. Tabela de demonstracao

A tabela abaixo mostra algumas entradas do dataset. Cada linha representa um dia do canil.

In [5]:
demo_columns = [
    "date",
    "dogs_small",
    "dogs_medium",
    "dogs_large",
    "cats",
    "equine_vaccine_visits",
    "puppies",
    "adult_new_animals",
    "quarantine_animals",
    "cleaning_runs",
    "outbreak_alert",
    "feed_kg_consumed",
    "vaccine_doses_used",
    "bleach_liters_used",
]

history_df.loc[:, demo_columns].tail(12)

,date,dogs_small,dogs_medium,dogs_large,cats,equine_vaccine_visits,puppies,adult_new_animals,quarantine_animals,cleaning_runs,outbreak_alert,feed_kg_consumed,vaccine_doses_used,bleach_liters_used
408,2026-05-10,69,89,55,59,0,0,2,9,15,0,65.32,3,26.56
409,2026-05-11,69,89,55,57,1,0,2,6,15,0,65.21,1,25.90
410,2026-05-12,69,89,55,58,0,1,5,15,16,0,67.77,1,30.10
411,2026-05-13,68,88,54,59,0,0,3,10,15,0,69.70,2,26.37
412,2026-05-14,68,88,54,60,0,1,5,10,15,0,63.36,2,26.88
413,2026-05-15,70,90,55,60,0,2,6,20,16,0,62.10,2,29.10
414,2026-05-16,70,90,55,60,0,0,1,9,15,0,67.39,1,25.95
415,2026-05-17,70,90,55,60,0,0,1,10,15,0,64.38,1,24.76
416,2026-05-18,69,89,55,60,0,3,2,12,15,0,67.08,0,26.68
417,2026-05-19,70,90,55,60,0,2,4,11,15,0,67.58,0,27.17


## 6. Metricas de desempenho

Para avaliar o modelo, separamos os ultimos 30 dias como teste temporal.

O modelo treina nos dias anteriores e tenta prever esses ultimos 30 dias.

Metricas usadas:

- **MAE**: erro absoluto medio. Mais facil de interpretar na unidade real do item.
- **RMSE**: penaliza mais erros grandes.
- **MAPE**: erro percentual medio.
- **R2**: quanto da variacao do consumo o modelo consegue explicar.

In [6]:
TARGETS = {
    "Racao (kg)": "feed_kg_consumed",
    "Vacinas (doses)": "vaccine_doses_used",
    "Agua sanitaria (litros)": "bleach_liters_used",
}


def evaluate_regression(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    errors = y_true - y_pred
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    non_zero = y_true != 0
    mape = np.mean(np.abs(errors[non_zero] / y_true[non_zero])) * 100 if non_zero.any() else np.nan
    ss_res = np.sum(errors ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot else np.nan
    return mae, rmse, mape, r2


train_df = history_df.iloc[:-30].copy()
test_df = history_df.iloc[-30:].copy()

metrics = []
prediction_frames = []

for label, target in TARGETS.items():
    model = InventoryConsumptionModel(alpha=2.0)
    model.fit(train_df, target_column=target)
    predictions = model.predict(test_df)
    mae, rmse, mape, r2 = evaluate_regression(test_df[target], predictions)

    metrics.append(
        {
            "item": label,
            "MAE": round(mae, 2),
            "RMSE": round(rmse, 2),
            "MAPE (%)": round(mape, 2),
            "R2": round(r2, 3),
        }
    )

    prediction_frames.append(
        pd.DataFrame(
            {
                "date": test_df["date"].values,
                "item": label,
                "real": test_df[target].values,
                "previsto": predictions.values,
            }
        )
    )

metrics_df = pd.DataFrame(metrics)
predictions_df = pd.concat(prediction_frames, ignore_index=True)

metrics_df

,item,MAE,RMSE,MAPE (%),R2
0,Racao (kg),1.95,2.44,2.98,-0.169
1,Vacinas (doses),1.15,1.30,79.67,-0.436
2,Agua sanitaria (litros),0.59,0.74,2.16,0.660


## 7. Grafico: real x previsto

O grafico compara o consumo real do periodo de teste com o consumo previsto pelo modelo.

In [7]:
plot_df = predictions_df.melt(
    id_vars=["date", "item"],
    value_vars=["real", "previsto"],
    var_name="serie",
    value_name="consumo",
)

fig = px.line(
    plot_df,
    x="date",
    y="consumo",
    color="serie",
    facet_row="item",
    markers=True,
    title="Consumo real x previsto nos ultimos 30 dias",
)
fig.update_layout(height=850, legend_title_text="Serie")
fig.update_yaxes(matches=None)
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 8. Saida operacional: dias restantes e alerta de compra

Apos prever o consumo futuro, o pipeline calcula:

```text
dias_restantes = estoque_atual / consumo_previsto
```

Na implementacao real, o calculo considera o consumo previsto dia a dia, descontando o estoque ate acabar.

O status e definido assim:

- `critical`: dias restantes menor ou igual ao prazo de reposicao + margem de seguranca;
- `warning`: ainda nao esta critico, mas esta perto;
- `ok`: cobertura suficiente.

In [ ]:
payload = build_forecast_payload(history_df, future_df)
stock_forecast_df = pd.DataFrame(payload["stock_forecast"])

stock_forecast_df[
    [
        "label",
        "current_stock",
        "unit",
        "average_daily_consumption",
        "days_remaining",
        "replenishment_days",
        "status",
        "recommended_reorder_quantity",
    ]
]

## 9. Grafico: dias restantes por item

Este grafico mostra a informacao mais importante para o canil: qual item esta mais perto de acabar.

In [ ]:
fig = px.bar(
    stock_forecast_df,
    x="label",
    y="days_remaining",
    color="status",
    text="days_remaining",
    title="Dias restantes de estoque por item",
    labels={"label": "Item", "days_remaining": "Dias restantes", "status": "Status"},
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_range=[0, max(30, stock_forecast_df["days_remaining"].max() * 1.15)])
fig.show()

## 10. Conclusao

Este modelo foi escolhido por ser simples, explicavel e util para uma primeira versao do sistema.

A maior entrega de valor para o canil e o alerta operacional:

- o que esta perto de acabar;
- quando pode acabar;
- quanto comprar;
- qual item priorizar.

Quando o sistema tiver dados reais de estoque e consumo, o mesmo notebook pode ser usado para comparar o desempenho do baseline com modelos mais avancados.